# Phase 1 - Résolution interactive du problème de supply chain

## Problème primal : Achat des lots chez MALANDRIN

**Variables :**  
- \( x_1 \): quantité de Lot 1  
- \( x_2 \): quantité de Lot 2  
- \( x_3 \): quantité de Lot 3  

**Objectif :** Minimiser le coût total  
**Contraintes :** Satisfaire les demandes minimales en fusils, grenades, chars, mitrailleuses et bazookas.

In [1]:
import pulp  # Bibliothèque pour résoudre des programmes linéaires

# ====================== CRÉATION DU PROBLÈME ======================
# On crée un problème de minimisation
prob = pulp.LpProblem("Achat_Lots_MALANDRIN", pulp.LpMinimize)

# ====================== VARIABLES DE DÉCISION ======================
# Variables continues (car les lots sont fractionnables)
x1 = pulp.LpVariable("Lots_1", lowBound=0)   # quantité de Lot 1
x2 = pulp.LpVariable("Lots_2", lowBound=0)   # quantité de Lot 2
x3 = pulp.LpVariable("Lots_3", lowBound=0)   # quantité de Lot 3

# ====================== FONCTION OBJECTIF ======================
# Coût total à minimiser (en millions de dollars)
prob += 10 * x1 + 12 * x2 + 15 * x3, "Cout_total"

# ====================== CONTRAINTES ======================
# Chaque contrainte correspond à une demande minimale

prob += 500 * x1 + 300 * x2 + 800 * x3 >= 100000, "Fusils"
prob += 1000 * x1 + 2000 * x2 + 1500 * x3 >= 200000, "Grenades"
prob += 10 * x1 + 20 * x2 + 15 * x3 >= 100, "Chars"
prob += 100 * x1 + 80 * x2 + 15 * x3 >= 400, "Mitrailleuses"
prob += 80 * x1 + 120 * x2 + 200 * x3 >= 400, "Bazookas"

# ====================== RÉSOLUTION ======================
# On utilise explicitement le solveur CBC avec un temps limite et moins de logs
status = prob.solve(pulp.PULP_CBC_CMD(msg=1, timeLimit=30, options=['sec 30']))

# ====================== AFFICHAGE DES RÉSULTATS ======================
print("Statut de la solution :", pulp.LpStatus[prob.status])
print("Coût total optimal :", pulp.value(prob.objective), "millions de dollars")

print("\nQuantités optimales de lots :")
print("Lot 1 :", x1.varValue)
print("Lot 2 :", x2.varValue)
print("Lot 3 :", x3.varValue)

Statut de la solution : Optimal
Coût total optimal : 1930.4347764000001 millions de dollars

Quantités optimales de lots :
Lot 1 : 0.0
Lot 2 : 8.6956522
Lot 3 : 121.73913


In [2]:
# ====================== VÉRIFICATION DES CONTRAINTES ======================
print("=== Vérification des demandes minimales ===")

fusils = 500*x1.varValue + 300*x2.varValue + 800*x3.varValue
grenades = 1000*x1.varValue + 2000*x2.varValue + 1500*x3.varValue
chars = 10*x1.varValue + 20*x2.varValue + 15*x3.varValue
mitrailleuses = 100*x1.varValue + 80*x2.varValue + 15*x3.varValue
bazookas = 80*x1.varValue + 120*x2.varValue + 200*x3.varValue

print(f"Fusils      : {fusils:,.0f}  >= 100000 ?")
print(f"Grenades    : {grenades:,.0f} >= 200000 ?")
print(f"Chars       : {chars:,.0f}     >= 100 ?")
print(f"Mitrailleuses : {mitrailleuses:,.0f}   >= 400 ?")
print(f"Bazookas    : {bazookas:,.0f}    >= 400 ?")
print(f"\nCoût total  : {pulp.value(prob.objective):.2f} M$")

=== Vérification des demandes minimales ===
Fusils      : 100,000  >= 100000 ?
Grenades    : 200,000 >= 200000 ?
Chars       : 2,000     >= 100 ?
Mitrailleuses : 2,522   >= 400 ?
Bazookas    : 25,391    >= 400 ?

Coût total  : 1930.43 M$


## Représentation graphique des contraintes

Nous fixons \( x_1 = 0 \) (car la solution optimale ne l'utilise pas) et nous représentons les contraintes dans le plan \( (x_2, x_3) \).

In [3]:
import matplotlib.pyplot as plt
import numpy as np

# ====================== REPRÉSENTATION GRAPHIQUE AMÉLIORÉE ET CORRIGÉE ======================
fig, ax = plt.subplots(figsize=(12, 10))

# Domaine pour tracer les droites (plus fin et stable)
x2 = np.linspace(0, 150, 400)

# === Tracé des contraintes (droites d'égalité) ===
# Fusils : 300x2 + 800x3 = 100000
ax.plot(x2, (100000 - 300*x2)/800, 'b--', linewidth=2, label='Fusils ≥ 100 000')

# Grenades : 2000x2 + 1500x3 = 200000
ax.plot(x2, (200000 - 2000*x2)/1500, 'g--', linewidth=2, label='Grenades ≥ 200 000')

# Chars : 20x2 + 15x3 = 100
ax.plot(x2, (100 - 20*x2)/15, 'r--', linewidth=2, label='Chars ≥ 100')

# Mitrailleuses : 80x2 + 15x3 = 400
ax.plot(x2, (400 - 80*x2)/15, 'purple', linestyle='--', linewidth=2, label='Mitrailleuses ≥ 400')

# Bazookas : 120x2 + 200x3 = 400
ax.plot(x2, (400 - 120*x2)/200, 'darkorange', linestyle='--', linewidth=2, label='Bazookas ≥ 400')

# === Point solution optimale ===
ax.plot(8.70, 121.74, 'ro', markersize=12, label='Solution optimale')

# Annotation claire du point optimal
ax.annotate('Solution optimale\n'
            'x₂ = 8.70   x₃ = 121.74\n'
            'Coût = 1930.43 M$',
            xy=(8.70, 121.74),
            xytext=(35, 95),
            fontsize=11,
            arrowprops=dict(facecolor='red', shrink=0.05, width=2, headwidth=8),
            bbox=dict(boxstyle="round,pad=0.6", facecolor="white", edgecolor="red", alpha=0.95))

# === Mise en forme professionnelle ===
ax.set_xlabel('Quantité de Lot 2 (x₂)', fontsize=13)
ax.set_ylabel('Quantité de Lot 3 (x₃)', fontsize=13)
ax.set_title('Région admissible du problème primal (x₁ = 0 fixé)', 
             fontsize=15, pad=20, fontweight='bold')

ax.set_xlim(0, 145)
ax.set_ylim(0, 145)
ax.grid(True, linestyle='--', alpha=0.7)

# Légende bien placée et lisible
ax.legend(loc='upper right', fontsize=11, frameon=True, facecolor='white', edgecolor='gray')

# Optionnel : ajouter un léger remplissage de la région admissible (version simplifiée)
# On remplit au-dessus de la contrainte la plus restrictive près de la solution (Bazookas + Fusils)
ax.fill_between(x2, (400 - 120*x2)/200, 145, 
                where=(x2 <= 20), 
                color='lightgreen', alpha=0.25, label='Région admissible')

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## Problème dual : Détermination des prix unitaires chez DETAILIN

**Variables :** \( y_1, y_2, y_3, y_4, y_5 \) = prix unitaires des fusils, grenades, chars, mitrailleuses, bazookas (en M$).

**Objectif :** Maximiser le bénéfice total  
**Contraintes :** Chaque lot coûte au plus le même prix que chez MALANDRIN.

In [ ]:
# ====================== PROBLÈME DUAL ======================
dual = pulp.LpProblem("Prix_Unitaires_DETAILIN", pulp.LpMaximize)

# Variables de décision (prix unitaires)
y1 = pulp.LpVariable("Prix_Fusil", lowBound=0)
y2 = pulp.LpVariable("Prix_Grenade", lowBound=0)
y3 = pulp.LpVariable("Prix_Char", lowBound=0)
y4 = pulp.LpVariable("Prix_Mitrailleuse", lowBound=0)
y5 = pulp.LpVariable("Prix_Bazooka", lowBound=0)

# Fonction objectif (bénéfice total à maximiser)
dual += 100000 * y1 + 200000 * y2 + 100 * y3 + 400 * y4 + 400 * y5, "Benefice_total"

# Contraintes (prix des lots)
dual += 500 * y1 + 1000 * y2 + 10 * y3 + 100 * y4 + 80 * y5 <= 10, "Lot_1"
dual += 300 * y1 + 2000 * y2 + 20 * y3 + 80 * y4 + 120 * y5 <= 12, "Lot_2"
dual += 800 * y1 + 1500 * y2 + 15 * y3 + 15 * y4 + 200 * y5 <= 15, "Lot_3"

# Résolution
dual.solve(pulp.PULP_CBC_CMD(msg=False))

# Résultats
print("Statut du dual :", pulp.LpStatus[dual.status])
print("Bénéfice total maximal :", pulp.value(dual.objective), "millions de dollars")

print("\nPrix unitaires optimaux (en M$) :")
print("Fusil         :", y1.varValue)
print("Grenade       :", y2.varValue)
print("Char          :", y3.varValue)
print("Mitrailleuse  :", y4.varValue)
print("Bazooka       :", y5.varValue)

Statut du dual : Optimal
Bénéfice total maximal : 1930.43482 millions de dollars

Prix unitaires optimaux (en M$) :
Fusil         : 0.010434783
Grenade       : 0.0044347826
Char          : 0.0
Mitrailleuse  : 0.0
Bazooka       : 0.0


In [ ]:
import pandas as pd

# ====================== COMPARAISON PRIMAL / DUAL ======================
print("=== COMPARAISON PRIMAL vs DUAL ===\n")

data = {
    "Problème": ["Primal (MALANDRIN)", "Dual (DETAILIN)"],
    "Type": ["Minimisation du coût", "Maximisation du bénéfice"],
    "Valeur optimale": [pulp.value(prob.objective), pulp.value(dual.objective)],
    "Statut": [pulp.LpStatus[prob.status], pulp.LpStatus[dual.status]]
}

df_comparison = pd.DataFrame(data)
print(df_comparison.to_string(index=False))

print("\nPrix unitaires optimaux (DETAILIN) :")
prices = {
    "Arme": ["Fusil", "Grenade", "Char", "Mitrailleuse", "Bazooka"],
    "Prix unitaire (M$)": [y1.varValue, y2.varValue, y3.varValue, y4.varValue, y5.varValue]
}
print(pd.DataFrame(prices).to_string(index=False))

print(f"\nValeur optimale identique ? → {abs(pulp.value(prob.objective) - pulp.value(dual.objective)) < 1e-4}")

=== COMPARAISON PRIMAL vs DUAL ===

          Problème                     Type  Valeur optimale  Statut
Primal (MALANDRIN)     Minimisation du coût      1930.434776 Optimal
   Dual (DETAILIN) Maximisation du bénéfice      1930.434820 Optimal

Prix unitaires optimaux (DETAILIN) :
        Arme  Prix unitaire (M$)
       Fusil            0.010435
     Grenade            0.004435
        Char            0.000000
Mitrailleuse            0.000000
     Bazooka            0.000000

Valeur optimale identique ? → True


## Analyse de sensibilité : Variation du coût du Lot 1

Nous faisons varier le coût du Lot 1 entre 8 M$ et 15 M$ et nous observons l'impact sur :
- les quantités optimales de lots
- le coût total
- les prix unitaires du dual

In [ ]:
import numpy as np

# ====================== ANALYSE DE SENSIBILITÉ ======================
print("=== ANALYSE DE SENSIBILITÉ SUR LE PRIX DU LOT 1 ===\n")
print(f"{'Prix Lot 1 (M$)':<15} {'x1':<10} {'x2':<10} {'x3':<12} {'Coût total':<15} {'Statut':<10}")
print("-" * 80)

# Liste des prix à tester
prix_lot1_values = np.arange(8.0, 15.1, 0.5)

for prix in prix_lot1_values:
    # On recrée le problème primal à chaque itération avec le nouveau coût
    prob_sens = pulp.LpProblem("Sensibilite_Lot1", pulp.LpMinimize)
    
    x1 = pulp.LpVariable("Lots_1", lowBound=0)
    x2 = pulp.LpVariable("Lots_2", lowBound=0)
    x3 = pulp.LpVariable("Lots_3", lowBound=0)
    
    # Objectif avec le coût variable du Lot 1
    prob_sens += prix * x1 + 12 * x2 + 15 * x3, "Cout_total"
    
    # Contraintes inchangées
    prob_sens += 500 * x1 + 300 * x2 + 800 * x3 >= 100000, "Fusils"
    prob_sens += 1000 * x1 + 2000 * x2 + 1500 * x3 >= 200000, "Grenades"
    prob_sens += 10 * x1 + 20 * x2 + 15 * x3 >= 100, "Chars"
    prob_sens += 100 * x1 + 80 * x2 + 15 * x3 >= 400, "Mitrailleuses"
    prob_sens += 80 * x1 + 120 * x2 + 200 * x3 >= 400, "Bazookas"
    
    # Résolution silencieuse
    prob_sens.solve(pulp.PULP_CBC_CMD(msg=False))
    
    if pulp.LpStatus[prob_sens.status] == "Optimal":
        cout = pulp.value(prob_sens.objective)
        print(f"{prix:<15.1f} {x1.varValue:<10.3f} {x2.varValue:<10.3f} {x3.varValue:<12.3f} {cout:<15.2f} Optimal")
    else:
        print(f"{prix:<15.1f} {'-':<10} {'-':<10} {'-':<12} {'Infeasible':<15} {pulp.LpStatus[prob_sens.status]}")

=== ANALYSE DE SENSIBILITÉ SUR LE PRIX DU LOT 1 ===

Prix Lot 1 (M$) x1         x2         x3           Coût total      Statut    
--------------------------------------------------------------------------------
8.0             200.000    0.000      0.000        1600.00         Optimal
8.5             200.000    0.000      0.000        1700.00         Optimal
9.0             200.000    0.000      0.000        1800.00         Optimal
9.5             200.000    0.000      0.000        1900.00         Optimal
10.0            0.000      8.696      121.739      1930.43         Optimal
10.5            0.000      8.696      121.739      1930.43         Optimal
11.0            0.000      8.696      121.739      1930.43         Optimal
11.5            0.000      8.696      121.739      1930.43         Optimal
12.0            0.000      8.696      121.739      1930.43         Optimal
12.5            0.000      8.696      121.739      1930.43         Optimal
13.0            0.000      8.696      

### Interprétation des résultats d'analyse de sensibilité

- Lorsque le prix du Lot 1 est **inférieur ou égal à 9,5 M$** :  
  la solution optimale consiste à utiliser **uniquement** le Lot 1 avec \( x_1 = 200 \), \( x_2 = 0 \), \( x_3 = 0 \).  
  Le coût total diminue linéairement avec le prix du Lot 1.

- Lorsque le prix du Lot 1 est **supérieur ou égal à 10,0 M$** :  
  la solution reste **stable** : \( x_1 = 0 \), \( x_2 = 8.696 \), \( x_3 = 121.739 \).  
  Le coût total reste constant à **1930.43 M$**.

- Le coût total est **sensible à la baisse** du prix du Lot 1 (diminution significative en dessous de 10 M$),  
  mais **insensible à la hausse** dans la plage testée (jusqu'à 15 M$).